# 第 9 章实践：稀疏索引 + 稠密计算


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）分工三问是：数据访问 ______、操作 ______、输出位置 ______。
2. （选择）Select-then-Compute 的"选择段"通常依赖第 7 章的哪组模式？
   A. load/move/store<br/>B. Gather / 原子仲裁 / 计数<br/>C. matmul/matmul_acc
3. （判断）仅凭选中率 10%，就能断定每线程 1 候选的设计不合理。（对/错）

In [ ]:
!cat ./answer/09.05_objective_answer.txt

## 2. 编程实践一（简单）：稀疏行筛选算子

**任务**：输入 [1024, 64] FP32 与行打分 [1024] FP32，选出分数 > 0 的行，紧凑写入输出区（上界 1024 行），返回选中计数。SIMT 完成筛选与紧凑化（提示：grid-stride 每轮每线程处理 1 行，`if score > 0` 判断 + 竞争计数器 `atomic_add` 取写入位置）；计数器用 INT32，Host 在启动前清零。计数与输出在 Kernel 完成后读取；同时输出来源行号以便按源行校验。原子取号导致紧凑区顺序不确定，不直接按原输入顺序逐行比较。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：筛选 + 稠密后处理

**任务**：在实践一的输出区上接 SIMD 段：对选中行做"每行减去行均值"（复用第 3 章归约+广播模式，作用于 [count, 64] 的紧凑区，count 为运行时标量——用固定 rank + valid_shape/边界保护处理）。先限定**单 Vector 核**，完整筛选阶段结束且 GM 写入对后续 MTE2 可见后，再执行后处理；明确 V→S/GM 发布→MTE2 的等待及缓存条件，auto_mutex 不能自动覆盖 SIMT 的 GM 直写。多核扩展先拆成两个同 Stream Kernel 验证，再研究同 Kernel 全核同步协议；不能仅顺序写两个函数就视为交接完成。


In [ ]:
# 在此完成实践二
pass

## 4. 编程实践三（困难）：门控 top-k 块选择（第 11 章核心机制的行级预演）

**任务**：32 个块有有限 FP32 分数，选 top-8；按“分数降序、同分原始块号升序”输出索引。用 SIMT 为候选 i 计算 rank_i：比它分数高的数量，加上同分且块号更小的数量；rank_i<8 时写 sel_idx[rank_i]=i。解释排名为何唯一，并对比 O(n²) 排名、重复 max 的 O(kn) 与阈值筛选。单次 atomic_max 只能求最大值，不能求第 8 名。


In [ ]:
# 在此完成实践三
pass

In [ ]:
!cat ./answer/09.05_practice_answer.txt

### 完整参考实现与验收

提供同 Kernel 的 UB 筛选缩放、GM 紧凑筛选、SIMD 行中心化、稳定 Top8。实践二先交付“生产 Kernel 完成 → 读取 count → 后处理 Kernel”的可验收基线；单 Kernel 的 GM 发布/缓存融合仍是后续改造任务，不能把这个阶段版本称作已实现该优化。

[参考源码](./src/hybrid_examples.py) 可在 950 上从章节目录执行 `python src/hybrid_examples.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/hybrid_examples.py


## 5. 本节小结

三个实践从"筛选→紧凑化"到"选择+计算"再到"top-k 块仲裁"，完成了 Select-then-Compute 的完整阶梯——实践三的确定性排名正是第 11 章 Gated-KV 选择的行级版本。下一章进入综合实战一：SDPA。